# 03 — Infrastructure & Physical Shocks

Эта ветка собирает дополнительные источники для физического графа нефтяного рынка.

Источники:

1. **BOEM/BSEE ArcGIS** — offshore platforms, offshore pipelines, planning areas.
2. **BSEE Production by Platform** — месячная offshore production по платформам.
3. **NOAA/NHC HURDAT2** — 6-часовые траектории Atlantic tropical cyclones.
4. **Global Energy Monitor GOIT** — transmission oil pipelines и ownership/status.
5. **OpenStreetMap / Overpass** — gap-filling для refineries и oil storage/terminals.

Все raw-файлы сохраняются в `data/raw/infrastructure/`.


## 1. Environment


In [1]:
import sys
from pathlib import Path

import pandas as pd

print(sys.version)
print(sys.executable)

assert sys.version_info[:2] == (3, 12)
assert ".venv" in sys.executable


3.12.16 (main, Oct 26 2026, 13:59:40) [Clang 23.1.3 ]
/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/.venv/bin/python


## 2. Project paths


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError("Project root not found.")


ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw" / "infrastructure"
PROCESSED_DIR = ROOT / "data" / "processed" / "infrastructure"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(ROOT)


/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph


In [3]:
import truststore

truststore.inject_into_ssl()

# 3. BOEM/BSEE spatial infrastructure

Используем официальный ArcGIS FeatureServer:

- platforms — layer 0;
- pipelines — layer 2;
- planning areas — layer 20.

GeoJSON сохраняем как raw, а platform points дополнительно превращаем в Parquet.


In [4]:
from src.data.boem_client import BoemClient

boem = BoemClient()


In [5]:
platforms_geojson = boem.fetch_platforms(active_only=False)
pipelines_geojson = boem.fetch_pipelines()
planning_areas_geojson = boem.fetch_planning_areas()

boem.save_geojson(
    platforms_geojson,
    RAW_DIR / "boem_platforms.geojson",
)
boem.save_geojson(
    pipelines_geojson,
    RAW_DIR / "boem_pipelines.geojson",
)
boem.save_geojson(
    planning_areas_geojson,
    RAW_DIR / "boem_planning_areas.geojson",
)

print("platforms:", len(platforms_geojson["features"]))
print("pipelines:", len(pipelines_geojson["features"]))
print("planning areas:", len(planning_areas_geojson["features"]))


platforms: 1315
pipelines: 17411
planning areas: 24


In [6]:
platforms = boem.points_to_frame(platforms_geojson)

platforms.to_parquet(
    PROCESSED_DIR / "boem_platforms.parquet",
    index=False,
)

platforms.head()


,OBJECTID,COMPLEX_ID_NUM,INSTALL_DATE,REMOVAL_DATE,longitude,latitude
0,17497399,20005,946684800000,None,-90.125616,29.041021
1,17497403,20008,-378691200000,None,-90.120977,29.043401
2,17497407,20011,-347155200000,None,-90.116217,29.056402
3,17497412,20015,-220924800000,None,-89.819586,28.969793
4,17497416,20020,-63158400000,None,-89.967169,28.993841


# 4. BSEE production by platform

BSEE считает monthly production rates по платформам на основе OGOR-A.

Нам нужны:

- platform / complex ID;
- production month;
- operator;
- oil rate;
- producing wells;
- lease / area / block.

ZIP скачивается один раз и сохраняется как raw artifact.


In [7]:
from src.data.bsee_client import BseeClient

bsee = BseeClient()

archive = bsee.download_production_by_platform(
    RAW_DIR / "bsee_production_by_platform.zip"
)

extracted = bsee.extract_archive(
    archive,
    RAW_DIR / "bsee_production_by_platform",
)

print([path.name for path in extracted])


['mv_prod_by_platform_all.txt']


In [8]:
production_by_platform = bsee.load_production_by_platform(extracted)

print(production_by_platform.shape)
print(production_by_platform.columns.tolist())

production_by_platform.head()


(1696663, 17)
['complex_id_num', 'structure_number', 'area_code', 'block_number', 'lease_number', 'structure_name', 'install_date', 'removal_date', 'pf_operator', 'pf_operator_num', 'production_date', 'producing_wells', 'bopd', 'mcfpd', 'boepd', 'bwpd', 'region_code']


,complex_id_num,structure_number,area_code,block_number,lease_number,structure_name,install_date,removal_date,pf_operator,pf_operator_num,production_date,producing_wells,bopd,mcfpd,boepd,bwpd,region_code
0,693,1,EI,53,00479,D,2000-04-01,NaT,GOM Shelf LLC,2451,2014-07-31,0,0,0,0,0,G
1,693,1,EI,53,00479,D,2000-04-01,NaT,GOM Shelf LLC,2451,2014-08-31,0,0,0,0,0,G
2,693,1,EI,53,00479,D,2000-04-01,NaT,GOM Shelf LLC,2451,2014-09-30,0,0,0,0,0,G
3,693,1,EI,53,00479,D,2000-04-01,NaT,GOM Shelf LLC,2451,2014-10-31,0,0,0,0,0,G
4,693,1,EI,53,00479,D,2000-04-01,NaT,GOM Shelf LLC,2451,2014-11-30,0,0,0,0,0,G


In [9]:
production_by_platform.to_parquet(
    PROCESSED_DIR / "bsee_production_by_platform.parquet",
    index=False,
)


# 5. NOAA / NHC HURDAT2

Используем pinned Atlantic HURDAT2 release **1851–2025, release 2025-092326**.

Для проекта оставляем 2015–2025.

HURDAT2 содержит 6-часовые track points:

- storm ID/name;
- datetime;
- lat/lon;
- status;
- max sustained wind;
- pressure;
- wind radii.

2026 season пока не включаем в historical best-track dataset, потому что официальный HURDAT2 на текущую дату финализирован только по 2025.


In [10]:
from src.data.noaa_hurdat import NoaaHurdatClient

noaa = NoaaHurdatClient()

storms = noaa.fetch(
    start_year=2015,
    end_year=2025,
)

print(storms.shape)
storms.head()


(6223, 23)


,storm_id,storm_name,datetime,year,status,record_identifier,latitude,longitude,max_wind_kt,min_pressure_mb,...,r34_sw,r34_nw,r50_ne,r50_se,r50_sw,r50_nw,r64_ne,r64_se,r64_sw,r64_nw
0,AL012015,ANA,2015-05-06 06:00:00+00:00,2015,LO,,26.8,-79.2,25,1016,...,0,0,0,0,0,0,0,0,0,0
1,AL012015,ANA,2015-05-06 12:00:00+00:00,2015,LO,,28.2,-78.5,25,1015,...,0,0,0,0,0,0,0,0,0,0
2,AL012015,ANA,2015-05-06 18:00:00+00:00,2015,LO,,29.7,-77.8,25,1014,...,0,0,0,0,0,0,0,0,0,0
3,AL012015,ANA,2015-05-07 00:00:00+00:00,2015,LO,,30.8,-77.5,30,1012,...,0,0,0,0,0,0,0,0,0,0
4,AL012015,ANA,2015-05-07 06:00:00+00:00,2015,LO,,30.7,-77.8,35,1010,...,0,100,0,0,0,0,0,0,0,0


In [11]:
storms.to_parquet(
    RAW_DIR / "nhc_hurdat2_atlantic_2015_2025.parquet",
    index=False,
)

storms.groupby("year")["storm_id"].nunique()


year
2015    12
2016    16
2017    18
2018    16
2019    20
2020    31
2021    21
2022    16
2023    21
2024    18
2025    13
Name: storm_id, dtype: int64

# 6. OpenStreetMap / Overpass gap filling

OSM не используем как authoritative source.

Он нужен только чтобы дополнить:

- refineries;
- oil storage;
- terminals;

в ключевых регионах, если объекта нет в официальных справочниках.

Запрашиваем по регионам, а не всю страну одним тяжёлым запросом.


In [12]:
from src.data.overpass_client import OverpassClient

overpass = OverpassClient()

OSM_REGIONS = [
    "gulf_coast",
    "midcontinent",
    "west_coast",
    "east_coast",
]


In [13]:
refinery_frames = []

for region in OSM_REGIONS:
    frame = overpass.fetch_refineries(region)
    refinery_frames.append(frame)
    print(region, len(frame))

osm_refineries = pd.concat(
    refinery_frames,
    ignore_index=True,
).drop_duplicates(
    subset=["osm_type", "osm_id"],
)

osm_refineries.to_parquet(
    RAW_DIR / "osm_refineries.parquet",
    index=False,
)

osm_refineries.head()


gulf_coast 33


Overpass https://overpass-api.de/api/interpreter returned HTTP 502 (attempt 1/5)


midcontinent 25
west_coast 509
east_coast 21


,osm_type,osm_id,name,operator,latitude,longitude,category,region,tags
0,node,585369735,Lyondell-Citgo Houston Refinery,NaN,29.713241,-95.235027,refinery,gulf_coast,"{""industrial"": ""refinery"", ""landuse"": ""industr..."
1,node,11285169577,NaN,Warren Oil Company,29.375746,-98.476497,refinery,gulf_coast,"{""addr:city"": ""San Antonio"", ""addr:housenumber..."
2,node,12438024068,Wakefield Biochar,Wakefield Biochar,30.831881,-83.255042,refinery,gulf_coast,"{""addr:city"": ""Valdosta"", ""addr:housenumber"": ..."
3,node,13176197648,Bunge,NaN,29.910740,-90.198478,refinery,gulf_coast,"{""man_made"": ""works"", ""name"": ""Bunge"", ""produc..."
4,node,13389948388,ZXP Technologies,NaN,29.817751,-95.051985,refinery,gulf_coast,"{""addr:city"": ""Highlands"", ""addr:housenumber"":..."


In [14]:
storage_frames = []

for region in OSM_REGIONS:
    frame = overpass.fetch_oil_storage(region)
    storage_frames.append(frame)
    print(region, len(frame))

osm_oil_storage = pd.concat(
    storage_frames,
    ignore_index=True,
).drop_duplicates(
    subset=["osm_type", "osm_id"],
)

osm_oil_storage.to_parquet(
    RAW_DIR / "osm_oil_storage_terminals.parquet",
    index=False,
)

osm_oil_storage.head()


gulf_coast 1413


Overpass https://overpass-api.de/api/interpreter returned HTTP 502 (attempt 1/5)
Overpass https://overpass-api.de/api/interpreter returned HTTP 504 (attempt 2/5)


midcontinent 59121
west_coast 334


Overpass https://overpass-api.de/api/interpreter returned HTTP 502 (attempt 1/5)


east_coast 1614


,osm_type,osm_id,name,operator,latitude,longitude,category,region,tags
0,node,9357078250,NaN,NaN,27.774739,-97.473531,oil_storage_or_terminal,gulf_coast,"{""content"": ""oil"", ""man_made"": ""storage_tank"",..."
1,node,9357078251,NaN,NaN,27.774737,-97.473484,oil_storage_or_terminal,gulf_coast,"{""content"": ""oil"", ""man_made"": ""storage_tank"",..."
2,node,9357078252,NaN,NaN,27.774743,-97.473435,oil_storage_or_terminal,gulf_coast,"{""content"": ""oil"", ""man_made"": ""storage_tank"",..."
3,node,9357078269,NaN,NaN,27.772260,-97.483049,oil_storage_or_terminal,gulf_coast,"{""abandoned"": ""yes"", ""content"": ""oil"", ""man_ma..."
4,node,9357078270,NaN,NaN,27.772259,-97.483006,oil_storage_or_terminal,gulf_coast,"{""abandoned"": ""yes"", ""content"": ""oil"", ""man_ma..."


# 7. Global Energy Monitor GOIT

Используем **Global Oil Infrastructure Tracker (GOIT)** как основной источник
наземных transmission oil / NGL pipelines.

В `data/external/gem/` кладём оригинальный архив:

```text
gem-download.zip
```

Внутри него parser автоматически извлекает:

- pipeline attributes XLSX;
- pipeline GeoJSON из spatial ZIP;
- Global Energy Ownership Tracker XLSX.

Связь между таблицами строится по `ProjectID`.

`ProjectID` используется как primary key pipeline-level таблицы, а ownership
сохраняется отдельно как one-to-many relation.


In [16]:
from src.data.gem_goit import GemGoitParser

GEM_DIR = ROOT / "data" / "external" / "gem"
GEM_DIR.mkdir(parents=True, exist_ok=True)

gem_parser = GemGoitParser(GEM_DIR)

download_zip = GEM_DIR / "gem-download.zip"

if download_zip.exists():
    gem_files = gem_parser.prepare_download_zip(download_zip)
else:
    gem_files = gem_parser.discover_files()

gem_files


GemFiles(pipeline_workbook=PosixPath('/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/external/gem/GEM-GOIT-Oil-NGL-Pipelines-2026-06.xlsx'), pipeline_geojson=PosixPath('/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/external/gem/GEM-GOIT-Oil-NGL-Pipelines-2026-07-21.geojson'), ownership_workbook=PosixPath('/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/external/gem/Global-Energy-Ownership-Tracker-October-2026-V1.xlsx'))

## 7.1 Нормализованные GEM tables

Parser собирает:

- `pipelines_all` — все pipeline projects;
- `pipelines_us` — проекты, связанные с United States;
- `pipeline_ownership_all` — ownership paths для всех проектов;
- `pipeline_ownership_us` — ownership только для U.S. projects;
- `entities` — GEM entity master table;
- `entity_ownership` — ownership edges между компаниями.

Геометрия берётся из GeoJSON и присоединяется к pipeline attributes по `ProjectID`.


In [17]:
gem_tables = gem_parser.build_tables(gem_files)

for name, frame in gem_tables.items():
    print(name, frame.shape)


pipelines_all (1926, 69)
pipelines_us (421, 69)
pipeline_ownership_all (23996, 13)
pipeline_ownership_us (7595, 13)
entities (28741, 29)
entity_ownership (27792, 7)


## 7.2 U.S. pipelines

Основная таблица для нашего графа — `pipelines_us`.

Она содержит инфраструктурные атрибуты, capacity, route geometry и агрегированную
ownership-информацию.


In [18]:
gem_pipelines_us = gem_tables["pipelines_us"]

preview_columns = [
    "project_id",
    "pipeline_name",
    "segment_name",
    "fuel",
    "status",
    "countries_or_areas",
    "capacity_boed",
    "length_merged_km",
    "geometry_type",
    "route_has_geometry",
    "parent_count",
]

gem_pipelines_us[
    [
        column
        for column in preview_columns
        if column in gem_pipelines_us.columns
    ]
].head(20)


,project_id,pipeline_name,segment_name,fuel,status,countries_or_areas,capacity_boed,length_merged_km,geometry_type,route_has_geometry,parent_count
0,P2592,Alberta Clipper Oil Pipeline,"2014 Pump Upgrade Expansion Project, U.S.",Oil,operating,United States,120000.0,0.0,NaN,False,3.0
1,P2590,Alberta Clipper Oil Pipeline,"2015 Pump Upgrade Expansion Project, U.S.",Oil,operating,United States,230000.0,0.0,NaN,False,3.0
2,P2646,Avalon Oil Pipeline,Avalon Extension,Oil,operating,United States,100000.0,51.5,LineString,True,17.0
3,P5106,Enbridge Line 5 Oil Pipeline,Bad River Relocation Project,Oil,construction,United States,540000.0,66.14,NaN,False,3.0
4,P3155,Seahawk Oil Pipeline,Blackhawk Pipeline,Oil,cancelled,United States,570000.0,32.19,LineString,True,2.0
5,P3172,Diamond Oil Pipeline,Byhalia Connection Project,Oil,cancelled,United States,<NA>,78.86,LineString,True,19.0
6,P2662,Western Corridor Oil Pipeline System,Capacity expansion,Oil,cancelled,United States,70000.0,0.0,NaN,False,8.0
7,P6523,BANGL Pipeline,Capacity Expansion,NGL,construction,United States,50000.0,0.0,NaN,False,7.0
8,P6525,EPIC NGL Pipeline,Capacity Expansion,NGL,construction,United States,75000.0,0.0,NaN,False,11.0
9,P2528,Eagle Ford JV Pipeline System,Capacity Expansion,Oil,operating,United States,300000.0,0.0,NaN,False,19.0


## 7.3 Geometry coverage

Отсутствующая geometry не считается ошибкой.

GEM может содержать pipeline project в database, даже если mapped route пока
отсутствует. Такие проекты сохраняем как infrastructure nodes без spatial route.


In [19]:
geometry_coverage = (
    gem_tables["pipelines_all"]["route_has_geometry"]
    .value_counts(dropna=False)
    .rename_axis("has_geometry")
    .to_frame("pipelines")
)

geometry_coverage


,pipelines
has_geometry,
True,1555
False,371


## 7.4 Ownership coverage

Ownership остаётся отдельной one-to-many таблицей, потому что у одного pipeline
может быть несколько owners / parent companies и разные ownership shares.


In [20]:
us_pipeline_ids = set(
    gem_tables["pipelines_us"]["project_id"]
)

ownership_coverage = {
    "us_pipelines": len(us_pipeline_ids),
    "us_pipelines_with_ownership": (
        gem_tables["pipeline_ownership_us"]["project_id"]
        .nunique()
    ),
    "ownership_rows": len(
        gem_tables["pipeline_ownership_us"]
    ),
}

ownership_coverage


{'us_pipelines': 421,
 'us_pipelines_with_ownership': 383,
 'ownership_rows': 7595}

## 7.5 Сохранение GEM

Нормализованные таблицы сохраняются в `data/processed/`.


In [21]:
gem_parser.save_tables(
    gem_tables,
    PROCESSED_DIR,
)

print("Saved normalized GEM tables.")


Saved normalized GEM tables.


# 8. EIA annual refinery-capacity snapshots

- company / operator;
- state / site;
- PADD;
- atmospheric crude distillation capacity;
- annual active/shutdown status.


In [22]:
from src.data.refinery_capacity import RefineryCapacityDownloader

refinery_downloader = RefineryCapacityDownloader()

refinery_files = refinery_downloader.download_range(
    start_year=2016,
    end_year=2026,
    target_dir=RAW_DIR / "eia_refinery_capacity",
)

[path.name for path in refinery_files]


['refcap2016.xls',
 'refcap2017.xls',
 'refcap2018.xls',
 'refcap2019.xls',
 'refcap2020.xls',
 'refcap2021.xls',
 'refcap2022.xls',
 'refcap2023.xls',
 'refcap2024.xlsx',
 'refcap2025.xlsx',
 'refcap2026.xlsx']

In [23]:
import re
import zipfile

REFCAP_DIR = RAW_DIR / "eia_refinery_capacity"
REFCAP_ZIP = RAW_DIR / "eia_refinery_capacity.zip"

# Если есть ZIP — распаковываем.
# Если файлы уже лежат в папке, ничего не меняем.
if REFCAP_ZIP.exists():
    with zipfile.ZipFile(REFCAP_ZIP) as archive:
        archive.extractall(RAW_DIR)

refinery_files = sorted(
    [
        path
        for path in REFCAP_DIR.glob("refcap20*")
        if path.suffix.lower() in {".xls", ".xlsx"}
    ]
)

print("Found refinery files:", len(refinery_files))

for path in refinery_files:
    print(path.name)

Found refinery files: 11
refcap2016.xls
refcap2017.xls
refcap2018.xls
refcap2019.xls
refcap2020.xls
refcap2021.xls
refcap2022.xls
refcap2023.xls
refcap2024.xlsx
refcap2025.xlsx
refcap2026.xlsx


In [24]:
def snake_case(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def refinery_report_year(path: Path) -> int:
    match = re.search(r"refcap(20\d{2})", path.stem.lower())

    if match is None:
        raise ValueError(
            f"Could not determine report year from {path.name}"
        )

    return int(match.group(1))


def excel_engine(path: Path) -> str:
    if path.suffix.lower() == ".xls":
        return "xlrd"

    return "openpyxl"


def find_header_row(path: Path) -> int:
    preview = pd.read_excel(
        path,
        sheet_name=0,
        header=None,
        nrows=25,
        engine=excel_engine(path),
    )

    for row_index, row in preview.iterrows():
        values = {
            str(value).strip().upper()
            for value in row.dropna()
        }

        required = {
            "CORPORATION",
            "COMPANY_NAME",
            "PRODUCT",
            "QUANTITY",
        }

        if required.issubset(values):
            return int(row_index)

    raise ValueError(
        f"Could not find header row in {path.name}"
    )


def read_refinery_capacity_file(
    path: Path,
) -> pd.DataFrame:
    header_row = find_header_row(path)

    frame = pd.read_excel(
        path,
        sheet_name=0,
        header=header_row,
        engine=excel_engine(path),
    )

    frame.columns = [
        snake_case(str(column))
        for column in frame.columns
    ]

    frame = frame.dropna(
        how="all"
    ).reset_index(drop=True)

    frame["report_year"] = refinery_report_year(path)
    frame["source_file"] = path.name

    required_columns = {
        "corporation",
        "company_name",
        "state_name",
        "site",
        "padd",
        "product",
        "supply",
        "quantity",
    }

    missing = required_columns - set(frame.columns)

    if missing:
        raise ValueError(
            f"{path.name} is missing columns: "
            f"{sorted(missing)}"
        )

    frame["quantity"] = pd.to_numeric(
        frame["quantity"],
        errors="coerce",
    )

    frame["padd"] = pd.to_numeric(
        frame["padd"],
        errors="coerce",
    ).astype("Int64")

    return frame

In [25]:
refinery_frames = []

for path in refinery_files:
    frame = read_refinery_capacity_file(path)

    refinery_frames.append(frame)

    print(
        path.name,
        "rows:",
        len(frame),
        "year:",
        frame["report_year"].iloc[0],
    )

refinery_capacity_long = pd.concat(
    refinery_frames,
    ignore_index=True,
)

print("\nCombined shape:", refinery_capacity_long.shape)
print(
    "Years:",
    refinery_capacity_long["report_year"].min(),
    "->",
    refinery_capacity_long["report_year"].max(),
)

refinery_capacity_long.head()

/var/folders/yf/82c7fyfs1m3c7t3l4p06rzp00000gn/T/ipykernel_20397/1228847298.py:26: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  preview = pd.read_excel(
/var/folders/yf/82c7fyfs1m3c7t3l4p06rzp00000gn/T/ipykernel_20397/1228847298.py:60: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(
/var/folders/yf/82c7fyfs1m3c7t3l4p06rzp00000gn/T/ipykernel_20397/1228847298.py:26: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  preview = pd.read_excel(
/var/folders/yf/82c7fyfs1m3c7t3l4p06rzp00000gn/T/ipykernel_20397/1228847298.py:60: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='

refcap2016.xls rows: 3814 year: 2016
refcap2017.xls rows: 3813 year: 2017
refcap2018.xls rows: 3720 year: 2018
refcap2019.xls rows: 3672 year: 2019
refcap2020.xls rows: 3711 year: 2020
refcap2021.xls rows: 3516 year: 2021
refcap2022.xls rows: 3482 year: 2022


/var/folders/yf/82c7fyfs1m3c7t3l4p06rzp00000gn/T/ipykernel_20397/1228847298.py:26: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  preview = pd.read_excel(
/var/folders/yf/82c7fyfs1m3c7t3l4p06rzp00000gn/T/ipykernel_20397/1228847298.py:60: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(
/var/folders/yf/82c7fyfs1m3c7t3l4p06rzp00000gn/T/ipykernel_20397/1228847298.py:26: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  preview = pd.read_excel(
/var/folders/yf/82c7fyfs1m3c7t3l4p06rzp00000gn/T/ipykernel_20397/1228847298.py:60: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='

refcap2023.xls rows: 3430 year: 2023
refcap2024.xlsx rows: 3415 year: 2024
refcap2025.xlsx rows: 3379 year: 2025
refcap2026.xlsx rows: 3336 year: 2026

Combined shape: (39288, 13)
Years: 2016 -> 2026


,corporation,survey,period,company_name,rdist_label,state_name,site,padd,product,supply,quantity,report_year,source_file
0,ALON ISRAEL OIL COMPANY LTD,820,16,ALON BAKERSFIELD OPERATING INC,West Coast,California,BAKERSFIELD,5,"CAT HYDROCRACKING, GAS OIL","Downstream Charge Capacity, Current Year (barr...",14250,2016,refcap2016.xls
1,ALON ISRAEL OIL COMPANY LTD,820,16,ALON BAKERSFIELD OPERATING INC,West Coast,California,BAKERSFIELD,5,"CAT HYDROCRACKING, GAS OIL","Downstream Charge Capacity, Current Year (barr...",15000,2016,refcap2016.xls
2,ALON ISRAEL OIL COMPANY LTD,820,16,ALON BAKERSFIELD OPERATING INC,West Coast,California,BAKERSFIELD,5,"CAT HYDROCRACKING, GAS OIL","Downstream Charge Capacity, Next Year (barrels...",15000,2016,refcap2016.xls
3,ALON ISRAEL OIL COMPANY LTD,820,16,ALON BAKERSFIELD OPERATING INC,West Coast,California,BAKERSFIELD,5,CAT REFORMING: LOW PRESSURE,"Downstream Charge Capacity, Current Year (barr...",7600,2016,refcap2016.xls
4,ALON ISRAEL OIL COMPANY LTD,820,16,ALON BAKERSFIELD OPERATING INC,West Coast,California,BAKERSFIELD,5,CAT REFORMING: LOW PRESSURE,"Downstream Charge Capacity, Current Year (barr...",8000,2016,refcap2016.xls


In [26]:
refinery_year_summary = (
    refinery_capacity_long
    .groupby("report_year")
    .agg(
        rows=("quantity", "size"),
        companies=("company_name", "nunique"),
        sites=("site", "nunique"),
        states=("state_name", "nunique"),
    )
    .reset_index()
)

refinery_year_summary

,report_year,rows,companies,sites,states
0,2016,3814,96,116,30
1,2017,3813,97,115,30
2,2018,3720,98,112,30
3,2019,3672,99,112,30
4,2020,3711,100,114,31
5,2021,3516,97,109,31
6,2022,3482,98,110,31
7,2023,3430,95,108,30
8,2024,3415,99,109,30
9,2025,3379,100,108,30


In [27]:
capacity_rows = refinery_capacity_long[
    refinery_capacity_long["product"].isin(
        [
            "OPERATING CAPACITY",
            "TOTAL OPERABLE CAPACITY",
        ]
    )
    & refinery_capacity_long["supply"].str.contains(
        "Atmospheric Crude Distillation Capacity",
        case=False,
        na=False,
    )
].copy()

capacity_rows[
    [
        "report_year",
        "company_name",
        "state_name",
        "site",
        "padd",
        "product",
        "supply",
        "quantity",
    ]
].head(20)

,report_year,company_name,state_name,site,padd,product,supply,quantity
32,2016,ALON REFINING KROTZ SPRINGS INC,Louisiana,KROTZ SPRINGS,3,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,80000
33,2016,ALON REFINING KROTZ SPRINGS INC,Louisiana,KROTZ SPRINGS,3,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,83000
35,2016,ALON REFINING KROTZ SPRINGS INC,Louisiana,KROTZ SPRINGS,3,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,80000
36,2016,ALON REFINING KROTZ SPRINGS INC,Louisiana,KROTZ SPRINGS,3,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,83000
61,2016,ALON USA ENERGY INC,Texas,BIG SPRING,3,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,73000
62,2016,ALON USA ENERGY INC,Texas,BIG SPRING,3,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,74000
66,2016,ALON USA ENERGY INC,Texas,BIG SPRING,3,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,73000
67,2016,ALON USA ENERGY INC,Texas,BIG SPRING,3,TOTAL OPERABLE CAPACITY,Atmospheric Crude Distillation Capacity (barre...,74000
79,2016,AMERICAN REFINING GROUP INC,Pennsylvania,BRADFORD,1,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,11000
80,2016,AMERICAN REFINING GROUP INC,Pennsylvania,BRADFORD,1,OPERATING CAPACITY,Atmospheric Crude Distillation Capacity (barre...,11800


In [28]:
capacity_rows["capacity_basis"] = pd.NA

calendar_mask = capacity_rows["supply"].str.contains(
    "calendar day",
    case=False,
    na=False,
)

stream_mask = capacity_rows["supply"].str.contains(
    "stream day",
    case=False,
    na=False,
)

capacity_rows.loc[
    calendar_mask,
    "capacity_basis",
] = "calendar_day"

capacity_rows.loc[
    stream_mask,
    "capacity_basis",
] = "stream_day"

capacity_rows[
    [
        "product",
        "capacity_basis",
        "quantity",
    ]
].head()

,product,capacity_basis,quantity
32,OPERATING CAPACITY,calendar_day,80000
33,OPERATING CAPACITY,stream_day,83000
35,TOTAL OPERABLE CAPACITY,calendar_day,80000
36,TOTAL OPERABLE CAPACITY,stream_day,83000
61,OPERATING CAPACITY,calendar_day,73000


In [29]:
REFINERY_KEYS = [
    "report_year",
    "corporation",
    "company_name",
    "rdist_label",
    "state_name",
    "site",
    "padd",
]

available_keys = [
    column
    for column in REFINERY_KEYS
    if column in capacity_rows.columns
]

capacity_pivot = (
    capacity_rows
    .pivot_table(
        index=available_keys,
        columns=[
            "product",
            "capacity_basis",
        ],
        values="quantity",
        aggfunc="max",
    )
    .reset_index()
)

capacity_pivot.columns = [
    "_".join(
        str(part)
        for part in column
        if str(part) not in {"", "nan"}
    )
    if isinstance(column, tuple)
    else str(column)
    for column in capacity_pivot.columns
]

capacity_pivot.columns = [
    snake_case(column)
    for column in capacity_pivot.columns
]

capacity_pivot.head()

,report_year,corporation,company_name,rdist_label,state_name,site,padd,operating_capacity_calendar_day,operating_capacity_stream_day,total_operable_capacity_calendar_day,total_operable_capacity_stream_day
0,2016,ACCESS INDUSTRIES,HOUSTON REFINING LP,Texas Gulf Coast,Texas,HOUSTON,3,263776.0,302300.0,263776.0,302300.0
1,2016,ALON ISRAEL OIL COMPANY LTD,ALON REFINING KROTZ SPRINGS INC,Louisiana Gulf Coast,Louisiana,KROTZ SPRINGS,3,80000.0,83000.0,80000.0,83000.0
2,2016,ALON ISRAEL OIL COMPANY LTD,ALON USA ENERGY INC,Texas Inland,Texas,BIG SPRING,3,73000.0,74000.0,73000.0,74000.0
3,2016,ALON ISRAEL OIL COMPANY LTD,PARAMOUNT PETROLEUM CORPORATION,West Coast,California,PARAMOUNT,5,NaN,NaN,84500.0,90000.0
4,2016,AMERICAN REFINING GROUP INC,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,11000.0,11800.0,11000.0,11800.0


In [30]:
rename_map = {
    (
        "operating_capacity_calendar_day"
    ): "operating_capacity_bpcd",
    (
        "operating_capacity_stream_day"
    ): "operating_capacity_bpsd",
    (
        "total_operable_capacity_calendar_day"
    ): "total_operable_capacity_bpcd",
    (
        "total_operable_capacity_stream_day"
    ): "total_operable_capacity_bpsd",
}

refineries = capacity_pivot.rename(
    columns=rename_map
)

refineries.head()

,report_year,corporation,company_name,rdist_label,state_name,site,padd,operating_capacity_bpcd,operating_capacity_bpsd,total_operable_capacity_bpcd,total_operable_capacity_bpsd
0,2016,ACCESS INDUSTRIES,HOUSTON REFINING LP,Texas Gulf Coast,Texas,HOUSTON,3,263776.0,302300.0,263776.0,302300.0
1,2016,ALON ISRAEL OIL COMPANY LTD,ALON REFINING KROTZ SPRINGS INC,Louisiana Gulf Coast,Louisiana,KROTZ SPRINGS,3,80000.0,83000.0,80000.0,83000.0
2,2016,ALON ISRAEL OIL COMPANY LTD,ALON USA ENERGY INC,Texas Inland,Texas,BIG SPRING,3,73000.0,74000.0,73000.0,74000.0
3,2016,ALON ISRAEL OIL COMPANY LTD,PARAMOUNT PETROLEUM CORPORATION,West Coast,California,PARAMOUNT,5,NaN,NaN,84500.0,90000.0
4,2016,AMERICAN REFINING GROUP INC,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,11000.0,11800.0,11000.0,11800.0


In [31]:
if {
    "operating_capacity_bpcd",
    "total_operable_capacity_bpcd",
}.issubset(refineries.columns):
    refineries["operating_capacity_share"] = (
        refineries["operating_capacity_bpcd"]
        / refineries["total_operable_capacity_bpcd"]
    )

refineries.head()

,report_year,corporation,company_name,rdist_label,state_name,site,padd,operating_capacity_bpcd,operating_capacity_bpsd,total_operable_capacity_bpcd,total_operable_capacity_bpsd,operating_capacity_share
0,2016,ACCESS INDUSTRIES,HOUSTON REFINING LP,Texas Gulf Coast,Texas,HOUSTON,3,263776.0,302300.0,263776.0,302300.0,1.0
1,2016,ALON ISRAEL OIL COMPANY LTD,ALON REFINING KROTZ SPRINGS INC,Louisiana Gulf Coast,Louisiana,KROTZ SPRINGS,3,80000.0,83000.0,80000.0,83000.0,1.0
2,2016,ALON ISRAEL OIL COMPANY LTD,ALON USA ENERGY INC,Texas Inland,Texas,BIG SPRING,3,73000.0,74000.0,73000.0,74000.0,1.0
3,2016,ALON ISRAEL OIL COMPANY LTD,PARAMOUNT PETROLEUM CORPORATION,West Coast,California,PARAMOUNT,5,NaN,NaN,84500.0,90000.0,NaN
4,2016,AMERICAN REFINING GROUP INC,AMERICAN REFINING GROUP INC,Appalachian No. 1,Pennsylvania,BRADFORD,1,11000.0,11800.0,11000.0,11800.0,1.0


In [32]:
print("Rows:", len(refineries))
print(
    "Years:",
    refineries["report_year"].min(),
    "->",
    refineries["report_year"].max(),
)

print(
    "PADDs:",
    sorted(
        refineries["padd"]
        .dropna()
        .unique()
        .tolist()
    ),
)

print(
    "Missing total capacity:",
    refineries[
        "total_operable_capacity_bpcd"
    ].isna().sum(),
)

print(
    "Missing operating capacity:",
    refineries[
        "operating_capacity_bpcd"
    ].isna().sum(),
)

Rows: 1429
Years: 2016 -> 2026
PADDs: [1, 2, 3, 4, 5, 6]
Missing total capacity: 0
Missing operating capacity: 33


In [33]:
refineries.groupby(
    "report_year"
).agg(
    refinery_rows=("site", "size"),
    total_capacity_bpcd=(
        "total_operable_capacity_bpcd",
        "sum",
    ),
    operating_capacity_bpcd=(
        "operating_capacity_bpcd",
        "sum",
    ),
)

,refinery_rows,total_capacity_bpcd,operating_capacity_bpcd
report_year,,,
2016,137,18317036.0,18165136.0
2017,137,18617027.0,18314127.0
2018,132,18598497.0,18566897.0
2019,132,18802435.0,18692335.0
2020,133,19152485.0,18548985.0
2021,127,18304100.0,17896000.0
2022,128,18120210.0,17789010.0
2023,126,18060369.0,17684549.0
2024,127,18384228.0,18374628.0


In [34]:
refinery_capacity_long.to_parquet(
    PROCESSED_DIR
    / "eia_refinery_capacity_long_2016_2026.parquet",
    index=False,
)

refineries.to_parquet(
    PROCESSED_DIR
    / "eia_refineries_2016_2026.parquet",
    index=False,
)

print("Saved:")
print(
    PROCESSED_DIR
    / "eia_refinery_capacity_long_2016_2026.parquet"
)
print(
    PROCESSED_DIR
    / "eia_refineries_2016_2026.parquet"
)

Saved:
/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/infrastructure/eia_refinery_capacity_long_2016_2026.parquet
/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/infrastructure/eia_refineries_2016_2026.parquet


# 9. Что дает ветка?

После выполнения:

### Offshore nodes
- BOEM platforms;
- BSEE operator / production information.

### Offshore edges
- BOEM/BSEE pipeline geometries.

### Weather events
- NOAA tropical cyclone track points.

### Onshore network
- GEM transmission pipelines + ownership + company entities;
- OSM refinery/storage/terminal gap-fill.

Позже отдельный feature-builder свяжет:

```text
storm -> affects -> platform
platform -> produces -> crude
platform -> connects_to -> pipeline
pipeline -> connects -> region / terminal
refinery -> located_in -> PADD
news -> describes -> event
```

И рассчитает hurricane exposure по расстоянию до платформ и по production-weighted capacity.
